In [5]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")

In [6]:
import mlflow

logged_model = 'runs:/b7562a6d06684f4f85f90008d9d433d4/daily_model_original_LR_BTC-USD'

# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(logged_model)




In [7]:
import warnings
warnings.filterwarnings("ignore")
from mlflow import MlflowClient, set_tracking_uri
import mlflow
from typing import Tuple
from tqdm import tqdm
import pandas as pd
from datetime import datetime, timedelta
import mysql.connector
import pyarrow
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import argparse
import os


def train_test_split(
    df: pd.DataFrame,
    cutoff_date: datetime,
    target_column_name: str,
    ) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    train_data = df[df.id_date < cutoff_date].reset_index(drop=True)
    test_data = df[df.id_date >= cutoff_date].reset_index(drop=True)

    X_train = train_data.drop(columns=[target_column_name])
    y_train = train_data[target_column_name]
    X_test = test_data.drop(columns=[target_column_name])
    y_test = test_data[target_column_name]

    return X_train, y_train, X_test, y_test


def Daily_model(exchange):
    temporality = 'daily'

    connection = mysql.connector.connect(
        user='root',
        password='root',
        host='localhost',
        port=3306,
        database='Historical_Data'
    )
    print("MySQL DB Connected")
    
    cursor = connection.cursor()
    cursor.execute(f"SELECT * FROM FT_DAILY_DATA WHERE Exchange = '{exchange}'")

    results = cursor.fetchall()
    columns = [column[0] for column in cursor.description]

    df_original = pd.DataFrame(results, columns=columns)
    df = df_original
    df['id_date'] = pd.to_datetime(df['id_date'], format='%Y%m%d')
    df = df[['Open', 'High', 'Low', 'Close', 'Volume', 'id_date']]

    # Use the provided train_test_split function
    X_train, y_train, X_test, y_test = train_test_split(
        df,
        cutoff_date=datetime(2023, 6, 1, 0, 0, 0),
        target_column_name='Close'
    )

    X_train = X_train.apply(lambda col: col.astype(int) if col.name != 'id_date' else col)
    X_test = X_test.apply(lambda col: col.astype(int) if col.name != 'id_date' else col)
 
    X_train_only_numeric = X_train.drop(columns = 'id_date')
    X_test_only_numeric = X_test.drop(columns = 'id_date')
    
    y_train = y_train.astype(int)
    y_test = y_test.astype(int)

    
    
    return X_test_only_numeric


In [8]:
exchange = 'BTC-USD'
X_test_only_numeric = Daily_model(exchange)

MySQL DB Connected


In [9]:
X_test_only_numeric

,Open,High,Low,Volume
0,27218,27346,26672,14678970415
1,26825,27304,26575,14837415000
2,27252,27317,26958,8385597470
3,27075,27407,26968,9360912318
4,27123,27130,25445,21513292646
...,...,...,...,...
397,62844,63203,61753,20151616992
398,62034,62188,59419,29756701685
399,60147,60400,56778,41149609230
400,57023,57497,53717,55417544033


In [10]:
# Predict on a Pandas DataFrame.


data = X_test_only_numeric

import pandas as pd
predictions = loaded_model.predict(pd.DataFrame(data))

predictions 

array([26854.35042074, 26972.33935359, 26990.81821285, 27177.65722203,
       25926.89457196, 26860.63787237, 26496.72336855, 26566.07571301,
       26511.50666019, 25773.47166063, 25928.62696423, 25783.47129554,
       26097.33674003, 25255.44592729, 25352.13629337, 26050.2187019 ,
       26503.37804694, 26415.09322299, 26785.98891797, 28012.55307695,
       30369.53011249, 30104.78269875, 31081.34640293, 30378.55452749,
       30709.88822804, 30139.34592623, 30771.90175396, 30058.31701331,
       30568.41853066, 30501.27650582, 30386.09744503, 30394.77299364,
       31136.68469846, 30840.39115341, 30365.29623909, 30835.29902536,
       30159.44220982, 30052.80077568, 30136.43566645, 30684.38974055,
       30569.07741491, 30530.42294545, 31466.20549664, 30456.13056056,
       30209.22815526, 30145.43713054, 29821.12265716, 29702.26318977,
       29967.02703246, 29849.21452371, 29834.41013263, 29682.13660634,
       30107.10566925, 29212.5497238 , 29120.97786235, 29422.9040936 ,
      